# GraphCodeBERT phân loại 4 lớp Code Clone — không fine-tune GraphCodeBERT

Notebook này lấy phần đọc dữ liệu theo notebook `Semantic_Code_Clone.ipynb` của bạn:

- `data.jsonl`: chứa `idx` và `func`.
- `train_50000.txt`, `valid_50000.txt`, `test_50000.txt`: mỗi dòng có dạng `id1<TAB>id2<TAB>label`.

Khác với notebook fine-tune ban đầu, notebook này **không fine-tune GraphCodeBERT**. GraphCodeBERT chỉ được dùng để trích xuất embedding cố định. Phần được train chỉ là classifier nhỏ bên ngoài.

Bản này cũng bổ sung đo thời gian inference: classifier-only trên feature đã encode và end-to-end từ raw code qua GraphCodeBERT.

Lưu ý: notebook này dùng `microsoft/graphcodebert-base` như encoder đóng băng ở chế độ token-only. Bản này chưa truyền DFG trực tiếp vào kiến trúc GraphCodeBERT, nên phù hợp để so sánh baseline frozen encoder công bằng với CodeBERT, BERT và DistilBERT.


In [1]:
# Cài thư viện nếu chạy trên Colab/Kaggle
!pip -q install transformers scikit-learn pandas numpy tqdm joblib

In [2]:
import os
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


## 1. Khai báo đường dẫn dữ liệu

Phần này lấy trực tiếp logic từ notebook bạn gửi. Bạn chỉ cần sửa lại `BASE_DIR` nếu vị trí dữ liệu khác.

In [3]:
# Đường dẫn giống notebook Semantic_Code_Clone.ipynb của bạn
BASE_DIR = "/content/drive/MyDrive/MultiCloneBERT/MultiCloneBERT"

DATA_JSONL = "/kaggle/input/datasets/quannvh/datasemantic-clone/dataset (2)/dataset/data.jsonl"
TRAIN_FILE = "/kaggle/input/datasets/quannvh/datasemantic-clone/50000-20260403T163943Z-3-001/50000/train_50000.txt"
VALID_FILE = "/kaggle/input/datasets/quannvh/datasemantic-clone/50000-20260403T163943Z-3-001/50000/valid_50000.txt"
TEST_FILE  = "/kaggle/input/datasets/quannvh/datasemantic-clone/50000-20260403T163943Z-3-001/50000/test_50000.txt"

# Nếu muốn test nhanh trên một phần nhỏ dữ liệu, đặt số dòng tại đây.
# Đặt None để dùng toàn bộ dữ liệu.
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
MAX_TEST_SAMPLES = None

LABEL_NAMES = ["Type-1", "Type-2", "Type-3", "Type-4"]

for p in [DATA_JSONL, TRAIN_FILE, VALID_FILE, TEST_FILE]:
    print(p, "->", Path(p).exists())

/kaggle/input/datasets/quannvh/datasemantic-clone/dataset (2)/dataset/data.jsonl -> True
/kaggle/input/datasets/quannvh/datasemantic-clone/50000-20260403T163943Z-3-001/50000/train_50000.txt -> True
/kaggle/input/datasets/quannvh/datasemantic-clone/50000-20260403T163943Z-3-001/50000/valid_50000.txt -> True
/kaggle/input/datasets/quannvh/datasemantic-clone/50000-20260403T163943Z-3-001/50000/test_50000.txt -> True


## 2. Đọc `data.jsonl` và các file pair

`data.jsonl` được dùng để tạo dictionary `id_to_code`. Sau đó các file split được đọc để tạo `train_df`, `valid_df`, `test_df` gồm các cột:

- `id1`
- `id2`
- `code1`
- `code2`
- `label`

In [4]:
def load_id_to_code(data_jsonl):
    id_to_code = {}
    bad_lines = 0

    with open(data_jsonl, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading data.jsonl"):
            line = line.strip()
            if not line:
                continue
            try:
                js = json.loads(line)
                idx = str(js["idx"])
                func = str(js["func"])
                id_to_code[idx] = func
            except Exception:
                bad_lines += 1

    print("Số lượng function trong data.jsonl:", len(id_to_code))
    print("Số dòng lỗi trong data.jsonl:", bad_lines)
    return id_to_code


def read_pairs(pair_file, id_to_code, max_samples=None):
    samples = []
    skipped_missing_code = 0
    skipped_bad_format = 0
    skipped_bad_label = 0

    with open(pair_file, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Reading {Path(pair_file).name}"):
            line = line.strip()
            if not line:
                continue

            parts = line.split("\t")
            if len(parts) != 3:
                skipped_bad_format += 1
                continue

            id1, id2, label = parts
            id1 = str(id1)
            id2 = str(id2)

            if id1 not in id_to_code or id2 not in id_to_code:
                skipped_missing_code += 1
                continue

            try:
                label = int(label)
            except Exception:
                skipped_bad_label += 1
                continue

            if label not in [0, 1, 2, 3]:
                skipped_bad_label += 1
                continue

            samples.append({
                "id1": id1,
                "id2": id2,
                "code1": id_to_code[id1],
                "code2": id_to_code[id2],
                "label": label,
            })

            if max_samples is not None and len(samples) >= max_samples:
                break

    print(f"\nĐọc file: {pair_file}")
    print("Số sample hợp lệ:", len(samples))
    print("Bỏ qua do thiếu code:", skipped_missing_code)
    print("Bỏ qua do sai format:", skipped_bad_format)
    print("Bỏ qua do sai label:", skipped_bad_label)

    return samples

In [5]:
id_to_code = load_id_to_code(DATA_JSONL)

sample_keys = list(id_to_code.keys())[:3]
for k in sample_keys:
    print("ID:", k)
    print(id_to_code[k][:300])
    print("=" * 80)

Loading data.jsonl: 0it [00:00, ?it/s]

Số lượng function trong data.jsonl: 9126
Số dòng lỗi trong data.jsonl: 0
ID: 10000832
    public static void main(String[] args) {
        LogFrame.getInstance();
        for (int i = 0; i < args.length; i++) {
            String arg = args[i];
            if (arg.trim().startsWith(DEBUG_PARAMETER_NAME + "=")) {
                properties.put(DEBUG_PARAMETER_NAME, arg.trim().substrin
ID: 10005623
    public synchronized String getSerialNumber() {
        if (serialNum != null) return serialNum;
        final StringBuffer buf = new StringBuffer();
        Iterator it = classpath.iterator();
        while (it.hasNext()) {
            ClassPathEntry entry = (ClassPathEntry) it.next();
         
ID: 10005624
            public Object run() {
                try {
                    MessageDigest digest = MessageDigest.getInstance("SHA");
                    digest.update(buf.toString().getBytes());
                    byte[] data = digest.digest();
                    serialNum = new BASE

In [6]:
train_samples = read_pairs(TRAIN_FILE, id_to_code, max_samples=MAX_TRAIN_SAMPLES)
valid_samples = read_pairs(VALID_FILE, id_to_code, max_samples=MAX_VALID_SAMPLES)
test_samples  = read_pairs(TEST_FILE,  id_to_code, max_samples=MAX_TEST_SAMPLES)

train_df = pd.DataFrame(train_samples)
valid_df = pd.DataFrame(valid_samples)
test_df  = pd.DataFrame(test_samples)

print("Train / Valid / Test shape:")
print(train_df.shape, valid_df.shape, test_df.shape)

display(train_df.head())

Reading train_50000.txt: 0it [00:00, ?it/s]


Đọc file: /kaggle/input/datasets/quannvh/datasemantic-clone/50000-20260403T163943Z-3-001/50000/train_50000.txt
Số sample hợp lệ: 19628
Bỏ qua do thiếu code: 10372
Bỏ qua do sai format: 0
Bỏ qua do sai label: 0


Reading valid_50000.txt: 0it [00:00, ?it/s]


Đọc file: /kaggle/input/datasets/quannvh/datasemantic-clone/50000-20260403T163943Z-3-001/50000/valid_50000.txt
Số sample hợp lệ: 6541
Bỏ qua do thiếu code: 3459
Bỏ qua do sai format: 0
Bỏ qua do sai label: 0


Reading test_50000.txt: 0it [00:00, ?it/s]


Đọc file: /kaggle/input/datasets/quannvh/datasemantic-clone/50000-20260403T163943Z-3-001/50000/test_50000.txt
Số sample hợp lệ: 6494
Bỏ qua do thiếu code: 3506
Bỏ qua do sai format: 0
Bỏ qua do sai label: 0
Train / Valid / Test shape:
(19628, 5) (6541, 5) (6494, 5)


,id1,id2,code1,code2,label
0,8539546,13106834,private static void unpackEntry(File desti...,public void copyFilesIntoProject(HashMap<S...,3
1,364438,441377,"public void convert(File src, File dest) t...","public void convert(File src, File dest) t...",0
2,9597889,21319238,"private void copy(File source, File dest) ...",FileCacheInputStreamFountain(FileCacheInpu...,3
3,18731109,23273706,public static boolean encodeFileToFile(Str...,public String[][] getProjectTreeData() {\n...,3
4,17147420,21933390,public final String latestVersion() {\n ...,public static List<String> getServers() th...,2


In [7]:
print("Train label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValid label distribution:")
print(valid_df["label"].value_counts().sort_index())

print("\nTest label distribution:")
print(test_df["label"].value_counts().sort_index())

Train label distribution:
label
0    2534
1    6693
2    5251
3    5150
Name: count, dtype: int64

Valid label distribution:
label
0     856
1    2226
2    1728
3    1731
Name: count, dtype: int64

Test label distribution:
label
0     812
1    2234
2    1745
3    1703
Name: count, dtype: int64


## 3. Tải GraphCodeBERT và đóng băng toàn bộ tham số

GraphCodeBERT ở đây chỉ dùng để sinh embedding, không cập nhật trọng số trong quá trình train classifier.

In [8]:
MODEL_NAME = "microsoft/graphcodebert-base"
MAX_LENGTH = 256
ENCODE_BATCH_SIZE = 16

# AutoTokenizer/AutoModel sẽ tự tải tokenizer và trọng số từ Hugging Face khi chạy lần đầu.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
encoder.eval()

for p in encoder.parameters():
    p.requires_grad = False

num_total = sum(p.numel() for p in encoder.parameters())
num_trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(f"Total GraphCodeBERT params: {num_total:,}")
print(f"Trainable GraphCodeBERT params: {num_trainable:,}")

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total GraphCodeBERT params: 124,645,632
Trainable GraphCodeBERT params: 0


## 4. Hàm trích xuất embedding từ GraphCodeBERT

Notebook dùng mean pooling theo `attention_mask`, sau đó chuẩn hóa L2 vector embedding.

In [9]:
def format_seconds(seconds):
    seconds = float(seconds)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    if h > 0:
        return f"{h:d}h {m:02d}m {s:05.2f}s"
    if m > 0:
        return f"{m:d}m {s:05.2f}s"
    return f"{s:.2f}s"


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_hidden = last_hidden_state * mask
    summed = masked_hidden.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


@torch.no_grad()
def encode_code_batch(codes, batch_size=ENCODE_BATCH_SIZE):
    embeddings = []
    start_time = time.perf_counter()

    for i in tqdm(range(0, len(codes), batch_size), desc="Encoding code"):
        batch_codes = codes[i:i + batch_size]
        inputs = tokenizer(
            batch_codes,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)

        outputs = encoder(**inputs)
        emb = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
        emb = F.normalize(emb, p=2, dim=1)
        embeddings.append(emb.cpu().numpy())

    total_time = time.perf_counter() - start_time
    print("Thời gian encode GraphCodeBERT:", format_seconds(total_time))
    return np.vstack(embeddings)

## 5. Encode các đoạn code duy nhất

Để nhanh hơn, mỗi đoạn code chỉ được encode một lần. Sau đó embedding được cache trong dictionary `code_to_emb`.

In [10]:
all_code_series = pd.concat(
    [
        train_df["code1"], train_df["code2"],
        valid_df["code1"], valid_df["code2"],
        test_df["code1"],  test_df["code2"],
    ],
    ignore_index=True,
)

unique_codes = pd.unique(all_code_series.astype(str))
unique_codes = [str(x) for x in unique_codes]

print("Số đoạn code duy nhất:", len(unique_codes))

unique_embeddings = encode_code_batch(unique_codes)
code_to_emb = {code: emb for code, emb in zip(unique_codes, unique_embeddings)}

print("Embedding shape:", unique_embeddings.shape)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Số đoạn code duy nhất: 4973


Encoding code:   0%|          | 0/311 [00:00<?, ?it/s]

Thời gian encode GraphCodeBERT: 23m 21.07s
Embedding shape: (4973, 768)


## 6. Tạo đặc trưng cho từng cặp code

Với mỗi cặp `(code1, code2)`, vector đầu vào classifier gồm:

- embedding của `code1`
- embedding của `code2`
- `abs(e1 - e2)`
- `e1 * e2`
- cosine similarity
- Euclidean distance

In [11]:
def build_pair_features(dataframe, desc="Building pair features"):
    features = []

    for c1, c2 in tqdm(
        zip(dataframe["code1"], dataframe["code2"]),
        total=len(dataframe),
        desc=desc,
    ):
        e1 = code_to_emb[str(c1)]
        e2 = code_to_emb[str(c2)]

        abs_diff = np.abs(e1 - e2)
        elem_mul = e1 * e2
        cosine = np.array([
            np.dot(e1, e2) / (np.linalg.norm(e1) * np.linalg.norm(e2) + 1e-9)
        ])
        euclidean = np.array([np.linalg.norm(e1 - e2)])

        feat = np.concatenate([e1, e2, abs_diff, elem_mul, cosine, euclidean])
        features.append(feat)

    return np.vstack(features).astype(np.float32)


X_train = build_pair_features(train_df, desc="Train features")
y_train = train_df["label"].values.astype(np.int64)

X_valid = build_pair_features(valid_df, desc="Valid features")
y_valid = valid_df["label"].values.astype(np.int64)

X_test = build_pair_features(test_df, desc="Test features")
y_test = test_df["label"].values.astype(np.int64)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

Train features:   0%|          | 0/19628 [00:00<?, ?it/s]

Valid features:   0%|          | 0/6541 [00:00<?, ?it/s]

Test features:   0%|          | 0/6494 [00:00<?, ?it/s]

X_train: (19628, 3074) y_train: (19628,)
X_valid: (6541, 3074) y_valid: (6541,)
X_test : (6494, 3074) y_test : (6494,)


## 7. Chuẩn hóa feature và tạo DataLoader

Chỉ fit `StandardScaler` trên train, sau đó transform valid/test để tránh rò rỉ dữ liệu.

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_valid_scaled = scaler.transform(X_valid).astype(np.float32)
X_test_scaled  = scaler.transform(X_test).astype(np.float32)

TRAIN_BATCH_SIZE = 256

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train_scaled), torch.tensor(y_train)),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
)

valid_loader = DataLoader(
    TensorDataset(torch.tensor(X_valid_scaled), torch.tensor(y_valid)),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test_scaled), torch.tensor(y_test)),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
)

## 8. Classifier nhẹ bên ngoài GraphCodeBERT

Phần này là thành phần duy nhất được huấn luyện. GraphCodeBERT vẫn đóng băng hoàn toàn.

In [13]:
class CloneClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=4, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)


input_dim = X_train_scaled.shape[1]
classifier = CloneClassifier(input_dim=input_dim, hidden_dim=512, num_classes=4, dropout=0.2).to(DEVICE)

optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

print(classifier)
print("Trainable classifier params:", sum(p.numel() for p in classifier.parameters() if p.requires_grad))

CloneClassifier(
  (net): Sequential(
    (0): Linear(in_features=3074, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=256, out_features=4, bias=True)
  )
)
Trainable classifier params: 1706756


## 9. Huấn luyện classifier và đo thời gian từng epoch

Mỗi epoch sẽ in ra:

- train loss
- validation accuracy
- validation macro-F1
- thời gian chạy epoch
- tổng thời gian đã chạy

Cell này cũng định nghĩa thêm hàm đo tốc độ inference để dùng ở phần test.


In [14]:
def synchronize_device():
    """Đồng bộ CUDA để đo thời gian chính xác hơn khi chạy GPU."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def print_inference_speed(title, total_time, num_samples):
    num_samples = max(int(num_samples), 1)
    ms_per_sample = total_time * 1000.0 / num_samples
    samples_per_second = num_samples / max(total_time, 1e-9)

    print(title)
    print(f"  Tổng thời gian: {format_seconds(total_time)}")
    print(f"  Số mẫu: {num_samples:,}")
    print(f"  Trung bình: {ms_per_sample:.4f} ms/sample")
    print(f"  Throughput: {samples_per_second:.2f} samples/s")


def evaluate_model(model, data_loader, return_time=False):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    total_samples = 0

    if return_time:
        synchronize_device()
        infer_start_time = time.perf_counter()

    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = criterion(logits, yb)

            batch_size = yb.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    if return_time:
        synchronize_device()
        infer_time = time.perf_counter() - infer_start_time

    avg_loss = total_loss / max(total_samples, 1)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    if return_time:
        return avg_loss, acc, macro_f1, np.array(all_preds), np.array(all_labels), infer_time

    return avg_loss, acc, macro_f1, np.array(all_preds), np.array(all_labels)


EPOCHS = 20
best_valid_f1 = -1.0
best_state = None
train_start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.perf_counter()
    classifier.train()

    running_loss = 0.0
    total_samples = 0

    for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()
        logits = classifier(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        batch_size = yb.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

    train_loss = running_loss / max(total_samples, 1)
    valid_loss, valid_acc, valid_f1, _, _ = evaluate_model(classifier, valid_loader)

    epoch_time = time.perf_counter() - epoch_start_time
    total_time = time.perf_counter() - train_start_time

    if valid_f1 > best_valid_f1:
        best_valid_f1 = valid_f1
        best_state = {k: v.detach().cpu().clone() for k, v in classifier.state_dict().items()}
        best_mark = "* best"
    else:
        best_mark = ""

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"valid_loss={valid_loss:.4f} | "
        f"valid_acc={valid_acc:.4f} | "
        f"valid_macro_f1={valid_f1:.4f} | "
        f"epoch_time={format_seconds(epoch_time)} | "
        f"elapsed={format_seconds(total_time)} {best_mark}"
    )

print("\nTổng thời gian train classifier:", format_seconds(time.perf_counter() - train_start_time))
print("Best valid macro-F1:", best_valid_f1)

if best_state is not None:
    classifier.load_state_dict(best_state)


Epoch 1/20:   0%|          | 0/77 [00:00<?, ?it/s]

Epoch 01/20 | train_loss=0.4571 | valid_loss=0.3494 | valid_acc=0.8306 | valid_macro_f1=0.8259 | epoch_time=1.85s | elapsed=1.85s * best


Epoch 2/20:   0%|          | 0/77 [00:00<?, ?it/s]

Epoch 02/20 | train_loss=0.3080 | valid_loss=0.2898 | valid_acc=0.8581 | valid_macro_f1=0.8645 | epoch_time=1.73s | elapsed=3.58s * best


Epoch 3/20:   0%|          | 0/77 [00:00<?, ?it/s]

...

Epoch 18/20 | train_loss=0.0410 | valid_loss=0.5966 | valid_acc=0.8682 | valid_macro_f1=0.8742 | epoch_time=1.69s | elapsed=30.88s 


Epoch 19/20:   0%|          | 0/77 [00:00<?, ?it/s]

Epoch 19/20 | train_loss=0.0407 | valid_loss=0.5946 | valid_acc=0.8717 | valid_macro_f1=0.8775 | epoch_time=1.72s | elapsed=32.59s 


Epoch 20/20:   0%|          | 0/77 [00:00<?, ?it/s]

Epoch 20/20 | train_loss=0.0392 | valid_loss=0.5843 | valid_acc=0.8681 | valid_macro_f1=0.8728 | epoch_time=1.69s | elapsed=34.29s 

Tổng thời gian train classifier: 34.29s
Best valid macro-F1: 0.8836230797058607


## 10. Đánh giá trên test set và đo thời gian inference classifier

Phần này đo inference trên `X_test_scaled`, tức là **feature đã được GraphCodeBERT encode sẵn**. Thời gian này phản ánh tốc độ của classifier bên ngoài, không bao gồm thời gian tokenizer/GraphCodeBERT.


In [15]:
test_loss, test_acc, test_f1, test_preds, test_labels, test_infer_time = evaluate_model(
    classifier,
    test_loader,
    return_time=True,
)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)
print("Test macro-F1:", test_f1)
print()
print_inference_speed(
    "Classifier-only inference trên test set, dùng feature đã encode sẵn:",
    test_infer_time,
    len(test_labels),
)
print()
print(classification_report(test_labels, test_preds, target_names=LABEL_NAMES, digits=4))
print("Confusion matrix:")
print(confusion_matrix(test_labels, test_preds))


Test loss: 0.37961657056633713
Test accuracy: 0.8694179242377579
Test macro-F1: 0.876342492858047

Classifier-only inference trên test set, dùng feature đã encode sẵn:
  Tổng thời gian: 0.19s
  Số mẫu: 6,494
  Trung bình: 0.0285 ms/sample
  Throughput: 35040.12 samples/s

              precision    recall  f1-score   support

      Type-1     0.9963    0.9926    0.9944       812
      Type-2     0.9920    0.9991    0.9955      2234
      Type-3     0.7532    0.7713    0.7622      1745
      Type-4     0.7658    0.7410    0.7532      1703

    accuracy                         0.8694      6494
   macro avg     0.8768    0.8760    0.8763      6494
weighted avg     0.8690    0.8694    0.8691      6494

Confusion matrix:
[[ 806    6    0    0]
 [   2 2232    0    0]
 [   1   12 1346  386]
 [   0    0  441 1262]]


## 11. Đo thời gian inference end-to-end từ raw code

Phần này đo thời gian từ cặp `code1`, `code2` thô đến dự đoán cuối cùng, bao gồm:

- tokenize code
- encode bằng GraphCodeBERT đã đóng băng
- tạo pair feature
- chuẩn hóa bằng scaler
- chạy classifier

Mặc định benchmark 1000 mẫu đầu của test set để chạy nhanh hơn. Đặt `INFER_BENCHMARK_SAMPLES = None` nếu muốn đo toàn bộ test set.


In [16]:
INFER_BENCHMARK_SAMPLES = min(1000, len(test_df))
# INFER_BENCHMARK_SAMPLES = None  # Bỏ comment nếu muốn đo toàn bộ test set
RAW_INFER_BATCH_SIZE = ENCODE_BATCH_SIZE


@torch.no_grad()
def predict_raw_pairs_with_timing(dataframe, batch_size=RAW_INFER_BATCH_SIZE):
    encoder.eval()
    classifier.eval()

    preds_all = []
    labels_all = []

    synchronize_device()
    start_time = time.perf_counter()

    for start_idx in tqdm(range(0, len(dataframe), batch_size), desc="Raw-code inference"):
        batch_df = dataframe.iloc[start_idx:start_idx + batch_size]
        code_a = batch_df["code1"].astype(str).tolist()
        code_b = batch_df["code2"].astype(str).tolist()

        inputs_a = tokenizer(
            code_a,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        inputs_b = tokenizer(
            code_b,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)

        out_a = encoder(**inputs_a)
        out_b = encoder(**inputs_b)

        emb_a = mean_pooling(out_a.last_hidden_state, inputs_a["attention_mask"])
        emb_b = mean_pooling(out_b.last_hidden_state, inputs_b["attention_mask"])
        emb_a = F.normalize(emb_a, p=2, dim=1)
        emb_b = F.normalize(emb_b, p=2, dim=1)

        abs_diff = torch.abs(emb_a - emb_b)
        elem_mul = emb_a * emb_b
        cosine = F.cosine_similarity(emb_a, emb_b, dim=1).unsqueeze(1)
        euclidean = torch.norm(emb_a - emb_b, p=2, dim=1).unsqueeze(1)

        feat = torch.cat([emb_a, emb_b, abs_diff, elem_mul, cosine, euclidean], dim=1)
        feat_np = feat.cpu().numpy().astype(np.float32)
        feat_np = scaler.transform(feat_np).astype(np.float32)

        xb = torch.tensor(feat_np, dtype=torch.float32, device=DEVICE)
        logits = classifier(xb)
        preds = torch.argmax(logits, dim=1)

        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(batch_df["label"].values.astype(np.int64))

    synchronize_device()
    total_time = time.perf_counter() - start_time

    return np.array(preds_all), np.array(labels_all), total_time


if INFER_BENCHMARK_SAMPLES is None:
    infer_df = test_df.reset_index(drop=True)
else:
    infer_df = test_df.head(INFER_BENCHMARK_SAMPLES).reset_index(drop=True)

raw_preds, raw_labels, raw_infer_time = predict_raw_pairs_with_timing(
    infer_df,
    batch_size=RAW_INFER_BATCH_SIZE,
)

raw_acc = accuracy_score(raw_labels, raw_preds)
raw_f1 = f1_score(raw_labels, raw_preds, average="macro")

print()
print("Raw-code benchmark accuracy:", raw_acc)
print("Raw-code benchmark macro-F1:", raw_f1)
print_inference_speed(
    "End-to-end raw-code inference, gồm tokenizer + GraphCodeBERT + classifier:",
    raw_infer_time,
    len(raw_labels),
)


Raw-code inference:   0%|          | 0/63 [00:00<?, ?it/s]


Raw-code benchmark accuracy: 0.864
Raw-code benchmark macro-F1: 0.8737670138314872
End-to-end raw-code inference, gồm tokenizer + GraphCodeBERT + classifier:
  Tổng thời gian: 9m 25.57s
  Số mẫu: 1,000
  Trung bình: 565.5699 ms/sample
  Throughput: 1.77 samples/s


## 12. Lưu classifier và scaler

File lưu ra không chứa trọng số fine-tuned của GraphCodeBERT vì GraphCodeBERT không được train. Khi inference lại, vẫn cần tải `microsoft/graphcodebert-base` để trích xuất embedding.

In [17]:
import joblib

SAVE_PATH = "graphcodebert_frozen_mlp_clone_classifier.pt"
SCALER_PATH = "graphcodebert_frozen_feature_scaler.joblib"

checkpoint = {
    "classifier_state_dict": classifier.state_dict(),
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "label_names": LABEL_NAMES,
    "input_dim": input_dim,
}

torch.save(checkpoint, SAVE_PATH)
joblib.dump(scaler, SCALER_PATH)

print("Saved classifier:", SAVE_PATH)
print("Saved scaler:", SCALER_PATH)

Saved classifier: graphcodebert_frozen_mlp_clone_classifier.pt
Saved scaler: graphcodebert_frozen_feature_scaler.joblib


## 13. Dự đoán thử một cặp code mới

In [18]:
@torch.no_grad()
def encode_single_code(code):
    inputs = tokenizer(
        [str(code)],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)

    outputs = encoder(**inputs)
    emb = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
    emb = F.normalize(emb, p=2, dim=1)
    return emb.squeeze(0).cpu().numpy()


def build_single_pair_feature(code_a, code_b):
    e1 = encode_single_code(code_a)
    e2 = encode_single_code(code_b)

    abs_diff = np.abs(e1 - e2)
    elem_mul = e1 * e2
    cosine = np.array([
        np.dot(e1, e2) / (np.linalg.norm(e1) * np.linalg.norm(e2) + 1e-9)
    ])
    euclidean = np.array([np.linalg.norm(e1 - e2)])

    feat = np.concatenate([e1, e2, abs_diff, elem_mul, cosine, euclidean])
    return feat.astype(np.float32).reshape(1, -1)


def predict_clone_type(code_a, code_b):
    """Dự đoán một cặp code và trả thêm thời gian inference end-to-end cho riêng cặp đó."""
    classifier.eval()
    encoder.eval()

    synchronize_device()
    start_time = time.perf_counter()

    feat = build_single_pair_feature(code_a, code_b)
    feat = scaler.transform(feat).astype(np.float32)
    xb = torch.tensor(feat).to(DEVICE)

    with torch.no_grad():
        logits = classifier(xb)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_id = int(np.argmax(probs))

    synchronize_device()
    infer_time = time.perf_counter() - start_time

    return {
        "label": LABEL_NAMES[pred_id],
        "probabilities": {LABEL_NAMES[i]: float(probs[i]) for i in range(len(LABEL_NAMES))},
        "inference_time_sec": float(infer_time),
        "inference_time_ms": float(infer_time * 1000.0),
    }


code_a = "int add(int a, int b) { return a + b; }"
code_b = "int sum(int x, int y) { return x + y; }"

predict_clone_type(code_a, code_b)


{'label': 'Type-2',
 'probabilities': {'Type-1': 2.6820802257245573e-21,
  'Type-2': 0.9983616471290588,
  'Type-3': 0.0016382999019697309,
  'Type-4': 4.984228158780475e-14},
 'inference_time_sec': 0.13523789000009856,
 'inference_time_ms': 135.23789000009856}

## Ghi chú dùng trong đồ án

Có thể mô tả thí nghiệm này như sau:

> GraphCodeBERT được sử dụng như một bộ trích xuất đặc trưng cố định cho từng đoạn mã nguồn. Toàn bộ tham số của GraphCodeBERT được đóng băng và không được cập nhật trong quá trình huấn luyện. Với mỗi cặp mã nguồn, embedding của hai đoạn mã được kết hợp thông qua phép nối trực tiếp, hiệu tuyệt đối, tích từng phần tử và một số độ đo tương đồng. Vector đặc trưng thu được được đưa vào một classifier nhẹ để phân loại bốn mức độ sao chép mã nguồn Type-1, Type-2, Type-3 và Type-4.